In [1]:
# Health check
import requests

SPACE_URL = "https://pagand-venra-haldet.hf.space"

response = requests.get(f"{SPACE_URL}/")
print(response.json())
# {"status": "VeNRA Hallucination Detector is Online", "device": "CPU", ...}

{'status': 'VeNRA Hallucination Detector is Online', 'device': 'CPU', 'adapter': 'pagand/venra', 'revision': 'r96', 'prompt_type': 'noinstruct', 'endpoints': {'POST /verify': 'Fast — single forward pass, one token read, no generation', 'POST /debug': 'Slow — full generation, returns model reasoning text'}}


In [2]:
payload = {
    "query":    "What was Apple's revenue in Q3 2023?",
    "context":  "Apple reported Q3 2023 revenue of $81.8 billion, down 1% year over year.",
    "trace":    "verification: Lookup.",
    "sentence": "Apple's revenue was $81.8 billion."
}

response = requests.post(f"{SPACE_URL}/verify", json=payload)
print(response.json())
# {"prediction": "SUPPORTED", "probabilities": {"supported": 0.94, "unfounded": 0.03, "general": 0.03}}

{'prediction': 'SUPPORTED', 'probabilities': {'supported': 1.0, 'unfounded': 0.0, 'general': 0.0}}


In [3]:
# Multiple sentences, same context — the real use case
sentences = [
    "Apple's Q3 2023 revenue was $81.8 billion.",        # should be SUPPORTED
    "Apple's Q3 2023 revenue grew 10% year over year.",  # should be UNFOUNDED
    "Revenue is a standard financial metric.",            # should be GENERAL
]

results = []
for sentence in sentences:
    payload["sentence"] = sentence
    r = requests.post(f"{SPACE_URL}/verify", json=payload)
    results.append({"sentence": sentence, **r.json()})

for r in results:
    print(f"[{r['prediction']}] {r['sentence']}")
    print(f"  probs: {r['probabilities']}\n")

[SUPPORTED] Apple's Q3 2023 revenue was $81.8 billion.
  probs: {'supported': 1.0, 'unfounded': 0.0, 'general': 0.0}

[UNFOUNDED] Apple's Q3 2023 revenue grew 10% year over year.
  probs: {'supported': 0.0, 'unfounded': 1.0, 'general': 0.0}

[GENERAL] Revenue is a standard financial metric.
  probs: {'supported': 0.033647, 'unfounded': 0.305376, 'general': 0.660977}



In [ ]:
# Debug call to inspect model reasoning

response = requests.post(f"{SPACE_URL}/debug", json=payload)
data = response.json()

print(data)
print(f"Prediction:  {data['prediction']}")
print(f"Probs:       {data['probabilities']}")
print(f"Analysis:\n{data['analysis']}")
# Analysis: Found
# Analysis: The evidence explicitly states Q3 2023 revenue of $81.8 billion...

{'prediction': 'GENERAL', 'probabilities': {'supported': 0.038141, 'unfounded': 0.31165, 'general': 0.650208}, 'analysis': " General\nAnalysis: The target sentence 'Revenue is a standard financial metric.' is classified as General because it reflects a universally known accounting principle that revenue represents a standard financial metric. While the specific numeric value of $81.8 billion is provided in the text, the statement regarding the nature of the metric remains a universally accepted financial fact. Furthermore, the Python trace correctly identifies this as a lookup verification, confirming the information is explicitly stated in the evidence. No contradictory or unsupported claims are present. Therefore, the classification as 'General' is appropriate. The query term 'Revenue' is a standard financial metric, making the statement a universally known financial axiom rather than a claim derived from the textual evidence. Hence, the sentence is classified as General. The trace c

In [ ]:
# verify
!curl -X POST "https://pagand-venra-haldet.hf.space/verify" \
  -H "Content-Type: application/json" \
  -d '{"query":"What was revenue?","context":"Revenue was $81.8B.","trace":"Found in table.","sentence":"Revenue was $81.8B."}'

# debug
!curl -X POST "https://pagand-venra-haldet.hf.space/debug" \
  -H "Content-Type: application/json" \
  -d '{"query":"What was revenue?","context":"Revenue was $81.8B.","trace":"Found in table.","sentence":"Revenue was $81.8B."}'

# health
!curl "https://pagand-venra-haldet.hf.space/"